In [8]:
import cobra
import pickle
from gem_utilities import media

In [2]:
model = cobra.io.read_sbml_model("model.xml")

In [9]:
# Load the media definitions
with open("test/test_files/media/media_definitions.pkl", "rb") as f:
    media_definitions = pickle.load(f)

In [4]:
model.metabolites.cpd00039_e0

Metabolite identifier,cpd00039_e0
Name,L-Lysine [e0]
Memory address,0x1183d3090
Formula,C6H15N2O2
Compartment,e0
In 2 reaction(s),"EX_cpd00039_e0, rxn08854_c0"


In [5]:
model.reactions.rxn08854_c0

Reaction identifier,rxn08854_c0
Name,L-Lysine transport via sodium symport
Memory address,0x11a71e490
Stoichiometry,cpd00039_e0 + cpd00971_e0 <=> cpd00039_c0 + cpd00971_c0 L-Lysine [e0] + Na+ [e0] <=> L-Lysine + Na+
GPR,
Lower bound,-1000.0
Upper bound,1000.0


In [6]:
model.reactions.EX_cpd00039_e0

Reaction identifier,EX_cpd00039_e0
Name,Exchange of L-Lysine [e0]
Memory address,0x11a728410
Stoichiometry,cpd00039_e0 --> L-Lysine [e0] -->
GPR,
Lower bound,0.0
Upper bound,1000.0


In [43]:
# Make a medium with lysine and set it
lysine_medium = media_definitions["marine_broth_wo_yeast_and_peptone"].copy()
lysine_medium["EX_cpd00039_e0"] = 10
model.medium = media.clean_media(model, lysine_medium)

/Users/helenscott/Documents/PhD/Segre-lab/GEM-utils/gem_utilities/media.py:31: UserWarning: Model does not have the exchange reaction EX_cpd09225_e0, so it was not set in the media.
  warnings.warn(
/Users/helenscott/Documents/PhD/Segre-lab/GEM-utils/gem_utilities/media.py:31: UserWarning: Model does not have the exchange reaction EX_cpd00242_e0, so it was not set in the media.
  warnings.warn(
/Users/helenscott/Documents/PhD/Segre-lab/GEM-utils/gem_utilities/media.py:31: UserWarning: Model does not have the exchange reaction EX_cpd00104_e0, so it was not set in the media.
  warnings.warn(
/Users/helenscott/Documents/PhD/Segre-lab/GEM-utils/gem_utilities/media.py:31: UserWarning: Model does not have the exchange reaction EX_cpd01826_e0, so it was not set in the media.
  warnings.warn(
/Users/helenscott/Documents/PhD/Segre-lab/GEM-utils/gem_utilities/media.py:31: UserWarning: Model does not have the exchange reaction EX_cpd00393_e0, so it was not set in the media.
  warnings.warn(
/User

In [12]:
model.optimize()

/Users/helenscott/Documents/PhD/Segre-lab/GEM-repos/GEM-mit1002/.venv/lib/python3.11/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


<Solution infeasible at 0x117c6a5d0>

In [ ]:
model.reactions.EX_cpd00039_e0

Reaction identifier,EX_cpd00039_e0
Name,Exchange of L-Lysine [e0]
Memory address,0x11a728410
Stoichiometry,cpd00039_e0 <=> L-Lysine [e0] <=>
GPR,
Lower bound,-1000
Upper bound,1000.0


In [16]:
model.reactions.rxn00062_c0.lower_bound = 0
model.optimize()

,fluxes,reduced_costs
rxn02201_c0,0.0,-6.260480e-19
rxn00351_c0,0.0,-7.825491e-17
rxn07431_c0,0.0,2.348160e-17
rxn00836_c0,0.0,-4.641031e-15
rxn00423_c0,0.0,2.430220e-14
...,...,...
rxn01993_c0,0.0,1.301083e-17
rxn02788_c0,0.0,-0.000000e+00
rxn01894_c0,0.0,4.088154e-18
rxn34493_c0,0.0,-3.682639e-18


In [22]:
# Define a metabolite for glutarate
glut = cobra.Metabolite(
    id="cpd00379_c0",
    name="Glutarate",
    formula="C5H6O4",
    charge=-2,
    compartment="c0",
)
glut.annotation = {"bigg.metabolite": "glutar",
                   "kegg.compound": "C00489",
                   "inchikey": "JFCQEDHGNNZCLN-UHFFFAOYSA-L"}
glut

Metabolite identifier,cpd00379_c0
Name,Glutarate
Memory address,0x11ae35290
Formula,C5H6O4
Compartment,c0
In 0 reaction(s),


In [23]:
# Add the metabolite to the model
model.add_metabolites(glut)
model.metabolites.cpd00379_c0

Metabolite identifier,cpd00379_c0
Name,Glutarate
Memory address,0x11ae35290
Formula,C5H6O4
Compartment,c0
In 0 reaction(s),


In [ ]:
# Define a reaction for rxn01729
glusd = cobra.Reaction("rxn01729_c0")
glusd.name = "glutarate-semialdehyde:NAD+ oxidoreductase"
glusd.add_metabolites({
    model.metabolites.cpd00001_c0: -1,
    model.metabolites.cpd00003_c0: -1,
    model.metabolites.cpd02089_c0: -1,
    model.metabolites.cpd00004_c0: 1,
    model.metabolites.cpd00067_c0: 2,
    model.metabolites.cpd00379_c0: 1
})

In [26]:
# Add the reaction to the model
model.add_reactions([glusd])
model.reactions.rxn01729_c0

Reaction identifier,rxn01729_c0
Name,glutarate-semialdehyde:NAD+ oxidoreductase
Memory address,0x11ac5df50
Stoichiometry,cpd00001_c0 + cpd00003_c0 + cpd02089_c0 --> cpd00004_c0 + 2 cpd00067_c0 + cpd00379_c0 H2O + NAD + 5-Oxopentanoate [c0] --> NADH + 2 H+ + Glutarate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [33]:
# =============================================================================
# Add new metabolites for the glutaryl-CoA branch of lysine catabolism
# =============================================================================

# Glutaryl-CoA (cpd00413) - 5C acyl-CoA, product of glutarate activation
glutcoa = cobra.Metabolite(
    id="cpd00413_c0",
    name="Glutaryl-CoA",
    formula="C26H37N7O19P3S",
    charge=-5,
    compartment="c0",
)
glutcoa.annotation = {
    "bigg.metabolite": "glutcoa",
    "kegg.compound": "C00527",
    "metacyc.compound": "GLUTARYL-COA",
    "inchikey": "SYKWLIJQEHRDNH-CKRMAKSASA-I",
    "seed.compound": "cpd00413",
}

# Crotonyl-CoA (cpd00650) - 4C trans-2-enoyl-CoA, β-oxidation intermediate
crotcoa = cobra.Metabolite(
    id="cpd00650_c0",
    name="Crotonyl-CoA",
    formula="C25H36N7O17P3S",
    charge=-4,
    compartment="c0",
)
crotcoa.annotation = {
    "bigg.metabolite": "b2coa",
    "kegg.compound": "C00877",
    "metacyc.compound": "CROTONYL-COA",
    "inchikey": "KFWWCMJSYSSPSK-BOGFJHSMSA-J",
    "seed.compound": "cpd00650",
}

# ETF-Oxidized (cpd27005) - electron transfer flavoprotein, oxidized form
# NOTE: formula contains R (generic) and mass is placeholder. ETF is a protein
# carrier that shuttles 2 e- between acyl-CoA dehydrogenases and the ETF:QO
# complex which reduces ubiquinone. You'll need ETF:QO to recycle ETF-red →
# ETF-ox or this branch will deadlock — see notes below.
etfox = cobra.Metabolite(
    id="cpd27005_c0",
    name="ETF-Oxidized",
    formula="C13H10N4O2R",
    charge=-1,
    compartment="c0",
)
etfox.annotation = {
    "metacyc.compound": "ETF-Oxidized",
    "seed.compound": "cpd27005",
}

# ETF-Reduced (cpd27006)
etfred = cobra.Metabolite(
    id="cpd27006_c0",
    name="ETF-Reduced",
    formula="C13H13N4O2R",
    charge=0,
    compartment="c0",
)
etfred.annotation = {
    "metacyc.compound": "ETF-Reduced",
    "seed.compound": "cpd27006",
}

# 3-Hydroxybutanoyl-CoA (cpd03043)
# NOTE: ModelSEED has this as the (3R) stereoisomer (InChIKey ...WZZMXTMRSA-J).
# The KEGG pathway map showed (S)-3-hydroxybutanoyl-CoA. The downstream
# dehydrogenase rxn03861's *name* says "(S)-..." but its stoichiometry uses
# this (R) compound, so the model is self-consistent internally. If you have a
# strong preference for the (S) form, the alternate SEED compound is cpd11874.
hbcoa = cobra.Metabolite(
    id="cpd03043_c0",
    name="3-Hydroxybutanoyl-CoA",
    formula="C25H38N7O18P3S",
    charge=-4,
    compartment="c0",
)
hbcoa.annotation = {
    "kegg.compound": "C05116",
    "metacyc.compound": "CPD-650",
    "inchikey": "QHHKKMYHDBRONY-WZZMXTMRSA-J",
    "seed.compound": "cpd03043",
}

model.add_metabolites([glutcoa, crotcoa, etfox, etfred, hbcoa])


# =============================================================================
# Add new reactions
# =============================================================================

# rxn01730 — Glutarate:CoA ligase (ADP-forming), EC 6.2.1.6
#   ATP + CoA + Glutarate <=> ADP + Pi + Glutaryl-CoA
#   ΔG° = -1.5 ± 0.34 kcal/mol → reversible is fine
gcl = cobra.Reaction("rxn01730_c0")
gcl.name = "Glutarate:CoA ligase (ADP-forming)"
gcl.add_metabolites({
    model.metabolites.cpd00002_c0: -1,  # ATP
    model.metabolites.cpd00010_c0: -1,  # CoA
    model.metabolites.cpd00379_c0: -1,  # Glutarate
    model.metabolites.cpd00008_c0: 1,   # ADP
    model.metabolites.cpd00009_c0: 1,   # Pi
    model.metabolites.cpd00413_c0: 1,   # Glutaryl-CoA
})
gcl.annotation = {
    "ec-code": "6.2.1.6",
    "bigg.reaction": "GLUTCOAs",
    "kegg.reaction": "R02402",
    "metacyc.reaction": "GLUTARATE--COA-LIGASE-RXN",
    "seed.reaction": "rxn01730",
}

# rxn19840 — Glutaryl-CoA dehydrogenase (ETF), EC 1.3.8.6
#   2 H+ + Glutaryl-CoA + ETF-ox <=> CO2 + Crotonyl-CoA + ETF-red
#   This is the decarboxylative step. SEED has no ΔG° (placeholder values) and
#   marks reversibility "?", but mechanistically the CO2 release makes this
#   physiologically irreversible forward. Setting lower_bound = 0 below.
gcdh = cobra.Reaction("rxn19840_c0")
gcdh.name = "Glutaryl-CoA dehydrogenase (ETF)"
gcdh.add_metabolites({
    model.metabolites.cpd00067_c0: -2,  # H+
    model.metabolites.cpd00413_c0: -1,  # Glutaryl-CoA
    model.metabolites.cpd27005_c0: -1,  # ETF-ox
    model.metabolites.cpd00011_c0: 1,   # CO2
    model.metabolites.cpd00650_c0: 1,   # Crotonyl-CoA
    model.metabolites.cpd27006_c0: 1,   # ETF-red
})
gcdh.lower_bound = 0  # irreversible (decarboxylation)
gcdh.annotation = {
    "ec-code": "1.3.8.6",
    "metacyc.reaction": "GLUTARYL-COA-DEHYDROGENASE-RXN",
    "seed.reaction": "rxn19840",
}

# rxn03874 — Crotonyl-CoA hydratase / (3R)-3-hydroxybutanoyl-CoA hydro-lyase
#   EC 4.2.1.17 (also 4.2.1.119, 4.2.1.55)
#   H2O + Crotonyl-CoA <=> 3-Hydroxybutanoyl-CoA
#   ΔG° = -1.02 ± 0.71 kcal/mol — reversible
crh = cobra.Reaction("rxn03874_c0")
crh.name = "Crotonyl-CoA hydratase"
crh.add_metabolites({
    model.metabolites.cpd00001_c0: -1,  # H2O
    model.metabolites.cpd00650_c0: -1,  # Crotonyl-CoA
    model.metabolites.cpd03043_c0: 1,   # 3-Hydroxybutanoyl-CoA
})
crh.annotation = {
    "ec-code": "4.2.1.17",
    "bigg.reaction": "3HBUTCOAH",
    "kegg.reaction": "R05595",
    "metacyc.reaction": "3-HYDROXBUTYRYL-COA-DEHYDRATASE-RXN",
    "seed.reaction": "rxn03874",
}

# rxn03861 — 3-Hydroxybutanoyl-CoA:NADP+ oxidoreductase, EC 1.1.1.157 / 1.1.1.36
#   NADP+ + 3-Hydroxybutanoyl-CoA <=> NADPH + H+ + Acetoacetyl-CoA
#   NOTE: this is the NADP+-dependent variant. The "classic" bacterial β-ox
#   3-hydroxyacyl-CoA dehydrogenase (FadB, EC 1.1.1.35) is NAD+-dependent. If
#   eggNOG annotated EC 1.1.1.35 specifically, the matching SEED reaction is
#   rxn00875 instead (NAD+ form). Worth double-checking which EC eggNOG gave.
hbcd = cobra.Reaction("rxn03861_c0")
hbcd.name = "(S)-3-Hydroxybutanoyl-CoA:NADP+ oxidoreductase"
hbcd.add_metabolites({
    model.metabolites.cpd00006_c0: -1,  # NADP+
    model.metabolites.cpd03043_c0: -1,  # 3-Hydroxybutanoyl-CoA
    model.metabolites.cpd00005_c0: 1,   # NADPH
    model.metabolites.cpd00067_c0: 1,   # H+
    model.metabolites.cpd00279_c0: 1,   # Acetoacetyl-CoA
})
hbcd.annotation = {
    "ec-code": "1.1.1.157",
    "kegg.reaction": "R05576",
    "metacyc.reaction": "RXN-5901",
    "seed.reaction": "rxn03861",
}

model.add_reactions([gcl, gcdh, crh, hbcd])

In [35]:
model.optimize()

,fluxes,reduced_costs
rxn02201_c0,0.018193,-9.536799e-19
rxn00351_c0,0.000000,-7.952466e-18
rxn07431_c0,0.000000,3.917850e-18
rxn00836_c0,0.000000,-1.621825e-02
rxn00423_c0,0.000000,-3.243651e-02
...,...,...
rxn01729_c0,292.989573,1.626693e-17
rxn01730_c0,292.989573,8.242642e-16
rxn19840_c0,0.000000,-1.619782e-13
rxn03874_c0,0.000000,1.547215e-17


In [34]:
# Export the model to json
cobra.io.save_json_model(model, "/Users/helenscott/Desktop/model.json")

In [36]:
# Save the model
cobra.io.write_sbml_model(model, "model.xml")

In [44]:
sol = cobra.flux_analysis.pfba(model)

In [45]:
sol.fluxes["rxn02167_c0"]

np.float64(-9.793552718442786)

In [46]:
sol.fluxes["rxn01451_c0"]

np.float64(9.793552718442786)

In [47]:
# Look for all fluxes that are above 100
for rxn_id, flux in sol.fluxes.items():
    if abs(flux) > 100:
        print(f"| {rxn_id} | {model.reactions.get_by_id(rxn_id).build_reaction_string(use_metabolite_names=True)} | {flux}")

In [48]:
# Save all the fluxes as a json file
import json
with open("/Users/helenscott/Desktop/fluxes.json", "w") as f:
    json.dump(sol.fluxes.to_dict(), f, indent=4)